# E039 — SR-MPGD : diagnostic du verrou + scan-aware

Objectif : dépasser le gagnant E038 (`hybrid r=.150`, 4 updates, γ=1000) sans sortir de la zone esthétique.

E039 compare 10 recettes sur le **même parent figé** et journalise chaque tentative de backtracking pour savoir si le vrai verrou est : rayon latent, LPIPS, core MAE ou non-amélioration de l'objectif.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image as DisplayImage, Markdown, display

RESULTS_DIR = Path(os.environ.get(
    "E039_RESULTS_DIR",
    "/data/e039-srmpgd-limiter-scanaware-v1",
))
print("E039 résultats :", RESULTS_DIR)


## 1. Verdict

In [ ]:
verdict_path = RESULTS_DIR / "verdict.json"
if not verdict_path.is_file():
    raise FileNotFoundError(
        f"E039 n'est pas terminé : {verdict_path} absent. "
        "Lancer d'abord bash scripts/run-e039-limiter-scanaware.sh"
    )
verdict = json.loads(verdict_path.read_text(encoding="utf-8"))
display(Markdown("**E039 terminé : résultats disponibles.**"))
display(verdict)


## 2. Classement complet — E038 contrôle + E039

In [ ]:
comparison = pd.read_csv(RESULTS_DIR / "method-comparison.csv")
preferred = [
    "method", "source", "profile", "max_iterations", "radius", "gamma",
    "qr_verify_exact_presets", "ssr", "original_exact",
    "full_module_error_count", "upstream_active_modules",
    "lpips", "latent_delta_rms", "clip_score", "clip_aesthetic", "hpsv2_1",
    "visual_guard_pass", "dominant_blocker", "accepted_updates", "rejected_all_iterations",
]
columns = [c for c in preferred if c in comparison.columns]
display(comparison[columns].sort_values(
    ["qr_verify_exact_presets", "lpips"],
    ascending=[False, True],
    na_position="last",
).reset_index(drop=True))


## 3. Quel garde-fou bloque réellement SR-MPGD ?

In [ ]:
blockers = pd.read_csv(RESULTS_DIR / "blocker-summary.csv")
display(blockers.sort_values(["profile", "radius", "max_iterations"]).reset_index(drop=True))

cols = [
    "rejected_by_latent_radius",
    "rejected_by_lpips_budget",
    "rejected_by_core_mae_budget",
    "rejected_by_objective_nonincrease",
]
plot_df = blockers.set_index("recipe")[cols]
ax = plot_df.plot(kind="bar", figsize=(16, 6))
ax.set_title("E039 — raisons de rejet des candidats SR-MPGD")
ax.set_ylabel("nombre de candidats rejetés")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


## 4. Toutes les images côte à côte

In [ ]:
sheet = RESULTS_DIR / "e039-all-methods-contact-sheet.png"
display(DisplayImage(filename=str(sheet), width=1500))


## 5. Chaque recette en grand

In [ ]:
e039_rows = comparison[comparison["source"] == "E039"].copy()
for _, row in e039_rows.sort_values(["profile", "radius", "max_iterations"]).iterrows():
    method = row["method"]
    final_path = RESULTS_DIR / method / "images" / f"iteration-{int(row['max_iterations']):03d}.png"
    title = (
        f"### {method} — SSR={int(row['qr_verify_exact_presets'])}/37 "
        f"MER={int(row['full_module_error_count'])}/841 LPIPS={row['lpips']:.4f} "
        f"safe={row['visual_guard_pass']} blocker={row.get('dominant_blocker')}"
    )
    display(Markdown(title))
    display(DisplayImage(filename=str(final_path), width=760))


## 6. SSR ↔ esthétique

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for profile, group in e039_rows.groupby("profile"):
    ax.scatter(group["lpips"], group["qr_verify_exact_presets"], label=profile)
    for _, row in group.iterrows():
        ax.annotate(str(int(row["max_iterations"])), (row["lpips"], row["qr_verify_exact_presets"]))
ax.set_xlabel("LPIPS vs parent")
ax.set_ylabel("QR-Verify exact presets / 37")
ax.set_title("E039 — SSR en fonction de la dérive perceptuelle")
ax.legend(title="profil")
plt.tight_layout()
plt.show()


## 7. Effet du nombre d'updates à rayon 0.150

In [ ]:
subset = e039_rows[e039_rows["radius"].round(3) == 0.150]
fig, ax = plt.subplots(figsize=(9, 6))
for profile, group in subset.groupby("profile"):
    group = group.sort_values("max_iterations")
    ax.plot(group["max_iterations"], group["qr_verify_exact_presets"], marker="o", label=profile)
ax.set_xlabel("updates SR-MPGD")
ax.set_ylabel("QR-Verify exact presets / 37")
ax.set_title("E039 — les updates supplémentaires augmentent-ils encore le SSR ?")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Trace détaillée du gagnant

In [ ]:
winner = verdict["research_winner"]
trace = pd.read_csv(RESULTS_DIR / winner / "trace.csv")
preferred_trace = [
    "iteration", "upstream_srl", "full_module_loss", "robust_loss", "lpips_loss",
    "objective", "full_module_error_count", "upstream_active_modules",
    "latent_gradient_rms", "raw_step_rms", "projected_step_rms",
    "accepted_step_rms", "accepted_alpha", "latent_delta_rms",
    "acceptance_reason", "rejected_trial_count",
]
display(trace[[c for c in preferred_trace if c in trace.columns]])

rejections = pd.read_csv(RESULTS_DIR / winner / "rejection-log.csv")
display(Markdown("### Dernières tentatives de backtracking du gagnant"))
display(rejections.tail(30))


## 9. E038 winner vs E039 winner

In [ ]:
control = RESULTS_DIR / "e038-control.json"
control_data = json.loads(control.read_text(encoding="utf-8"))
winner_row = e039_rows[e039_rows["method"] == winner].iloc[0].to_dict()
summary = pd.DataFrame([
    {
        "method": "E038 hybrid r=.150 i4",
        "SSR /37": control_data.get("qr_verify_exact_presets"),
        "MER": control_data.get("full_module_error_count"),
        "LPIPS": control_data.get("lpips"),
        "original_exact": control_data.get("original_exact"),
    },
    {
        "method": winner,
        "SSR /37": winner_row.get("qr_verify_exact_presets"),
        "MER": winner_row.get("full_module_error_count"),
        "LPIPS": winner_row.get("lpips"),
        "original_exact": winner_row.get("original_exact"),
    },
])
display(summary)
